# Phase 2A — Threshold & Scale Robustness

**Fast ℓp Proximity Operator | Numerical revision experiments**

This notebook evaluates **branch decisions and numerical accuracy** very close to the activation threshold, for both `float32` and `float64` and across three orders-of-magnitude scales of λ. It uses the existing Chebyshev, Newton and Bisection evaluators **without modifying them**.

It does **not** report GPU speedups; timing stability will be a separate Phase 2B notebook. Phase 1 results remain unchanged in `revision/phase1/`.

Mathematical convention: at $|y|=\tau_{p,\lambda}$, the proximity operator is set-valued $\{0,\operatorname{sign}(y)\rho_{p,\lambda}\}$. The diagnostic reference chooses the zero minimizer in this tie. The threshold oracle is computed at high precision and classifies each **actually representable** input independently of the approximate solver's threshold.

## A — Clone the experiment repository and install dependencies

This is GitHub-first; it works when opened directly from the public revision branch in Colab. Existing checkout is reused. A CPU runtime is sufficient for all correctness experiments.

In [ ]:
from pathlib import Path
import os, sys, subprocess

BRANCH = "revision/major-experiments"
URL = "https://github.com/Jy-U1/Fast_Lp_Prox.git"
ROOT = Path("/content/Fast_Lp_Prox") if Path("/content").is_dir() else Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(ROOT)], check=True)
if not (ROOT / "revision/phase2/scripts/run_threshold_robustness.py").exists():
    raise FileNotFoundError("Phase 2A source missing. Push the Phase 2A overlay to the revision branch first.")
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]", "mpmath", "matplotlib", "pandas"], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository:", ROOT)
print("Current Git commit:", commit)

## B — Fast numerical unit tests

This should run on CPU even if the runtime has no GPU. The tests cover the exact-tie convention, `nextafter` inputs, reference roots, and the actual preloaded solver API.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_phase2_threshold.py"], check=True)

## C — CPU smoke test (default)

Small degree-fitting grid for checking that the end-to-end computation and file output work. Smoke data must **not** be used for final whole-grid conclusions. The experiment includes negative/positive inputs and both sides of the exact threshold.

In [ ]:
from pathlib import Path
RESULTS = ROOT / "results/phase2"
SMOKE = RESULTS / "threshold_smoke"
cmd = [sys.executable, "revision/phase2/scripts/run_threshold_robustness.py",
       "--device", "cpu", "--p-grid", "1/3,0.999", "--lambda-grid", "1,1e6",
       "--dtypes", "float32,float64", "--fit-n", "512", "--validation-n", "1001",
       "--mp-dps", "70", "--ref-iterations", "140", "--output-dir", str(SMOKE), "--overwrite"]
subprocess.run(cmd, check=True)
subprocess.run([sys.executable, "revision/phase2/scripts/rebuild_threshold_artifacts.py",
                "--root", str(SMOKE)], check=True)

## D — Full threshold + scale experiment (opt-in)

Set `RUN_FULL=True` to collect the **formal Phase 2A data**. Typical Colab CPU/GPU works; GPU is used only for candidate forward evaluation. The independent mpmath oracle runs on CPU and is not timed. All six exponents, λ in {1e−6, 1, 1e6}, both dtypes, signed threshold neighbors and far-field points are included.

The full run may take several minutes. It only overwrites `results/phase2/threshold_formal`, never Phase 1.

In [ ]:
import torch
RUN_FULL = False       # Explicitly opt in after smoke checks.
FULL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FORMAL = RESULTS / "threshold_formal"
if RUN_FULL:
    full_cmd = [sys.executable, "revision/phase2/scripts/run_threshold_robustness.py",
                "--device", FULL_DEVICE,
                "--p-grid", "1/3,2/5,3/4,0.9,0.99,0.999",
                "--lambda-grid", "1e-6,1,1e6", "--dtypes", "float32,float64",
                "--fit-n", "8192", "--validation-n", "80001",
                "--mp-dps", "80", "--ref-iterations", "155",
                "--output-dir", str(FORMAL), "--overwrite"]
    subprocess.run(full_cmd, check=True)
    subprocess.run([sys.executable, "revision/phase2/scripts/rebuild_threshold_artifacts.py",
                    "--root", str(FORMAL)], check=True)
else:
    print("Full Phase 2A run is disabled. Analyzing the smoke results only.")

## E — Inspect the computed support decisions and accuracy

A support mismatch exactly adjacent to the mathematical threshold can produce a large **output** error even when the **objective** gap is tiny, because the scalar prox jumps from zero to a nonzero branch. The accompanying CSV records both quantities; do not suppress or silently correct those cases.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
ACTIVE = FORMAL if RUN_FULL else SMOKE
quality = pd.read_csv(ACTIVE / "threshold_quality_checks.csv")
summary = pd.read_csv(ACTIVE / "threshold_summary.csv")
thresholds = pd.read_csv(ACTIVE / "threshold_parameters.csv")
discrepancies = pd.read_csv(ACTIVE / "support_mismatch_details.csv")
print("ANALYSIS SOURCE:", ACTIVE)
display(quality)
display(summary[["p_label", "lambda", "dtype", "configuration", "tested_inputs",
                 "support_mismatch_count", "max_near_norm_output_error", "max_norm_stationarity_residual"]].head(25))
print("Near-threshold support disagreement records:", len(discrepancies))
if not discrepancies.empty:
    display(discrepancies[["p_label", "lambda", "dtype", "configuration", "y",
                           "oracle_side", "x_output", "oracle_x",
                           "threshold_gap_center_ulp", "norm_output_error",
                           "norm_objective_regret"]].head(15))
figure = ACTIVE / "figures/near_threshold_error.png"
if figure.exists():
    display(Image(filename=str(figure)))

## F — Export self-contained data ZIP for local `Revision/Archives`

Produces an archive with experiment code, version description, and available smoke/formal results. Keep the generated ZIP locally; do not commit the archive itself to Git. Publish validated individual source/data files as appropriate.

In [ ]:
import datetime, json, zipfile
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%SZ")
zip_path = ROOT / f"FastLp_Phase2A_Threshold_Export_{stamp}.zip"
sources = [ROOT / "revision/phase2/README.md",
           ROOT / "revision/phase2/scripts/run_threshold_robustness.py",
           ROOT / "revision/phase2/scripts/rebuild_threshold_artifacts.py",
           ROOT / "revision/phase2/notebooks/Phase2_Threshold_Robustness.ipynb",
           ROOT / "tests/test_phase2_threshold.py"]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in sources:
        z.write(f, f.relative_to(ROOT))
    z.writestr("phase2_run_commit.txt", f"Repository Git commit: {commit}\n")
    for d in (SMOKE, FORMAL):
        if d.exists():
            for f in d.rglob("*"):
                if f.is_file():
                    z.write(f, f.relative_to(ROOT))
print("Saved:", zip_path, "bytes:", zip_path.stat().st_size)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not in Colab; the ZIP is available at the path above.")